In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score
import nltk
from nltk.corpus import stopwords
import string

# Download stopwords if not already available
nltk.download('stopwords')

# Define a function to preprocess the text
def preprocess_text(text, stop_words):
    """Preprocess the input text by removing punctuation, converting to lowercase, and removing stopwords."""
    try:
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = ' '.join([word for word in text.split() if word not in stop_words])
        return text
    except Exception as e:
        print(f"Error during text preprocessing: {e}")
        return ""

# Load the dataset
def load_data(filepath):
    """Load data from a CSV file."""
    try:
        df = pd.read_csv(filepath)
        print(f"Data loaded successfully with shape: {df.shape}")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

# Main function to execute the pipeline
def main():
    # File path to dataset
    file_path = "/Users/reza/Desktop/ML/amazon/Reviews.csv"
    
    # Load the data
    df = load_data(file_path)
    if df is None:
        return

    # Check if required columns exist
    if 'Text' not in df.columns or 'Score' not in df.columns:
        print("Dataset does not contain the required columns: 'Text' and 'Score'")
        return

    # Preprocess text data
    stop_words = set(stopwords.words('english'))
    df['cleaned_review'] = df['Text'].apply(lambda x: preprocess_text(x, stop_words))

    # Define features and target
    X = df['cleaned_review']
    y = df['Score']

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Vectorize the text data
    tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    # Train a logistic regression model
    model = LogisticRegression(max_iter=1000, solver='liblinear')
    model.fit(X_train_tfidf, y_train)

    # Predict and evaluate the model
    y_pred = model.predict(X_test_tfidf)

    # Round and clip predictions to match target labels
    y_pred_classes = np.round(y_pred).astype(int)
    y_pred_classes = np.clip(y_pred_classes, y.min(), y.max())

    # Compute metrics
    mse = mean_squared_error(y_test, y_pred_classes)
    accuracy = accuracy_score(y_test, y_pred_classes)
    print(f"Mean Squared Error: {mse:.2f}")
    print(f"Accuracy: {accuracy:.2f}")

    # Hyperparameter tuning using RandomizedSearchCV
    param_distributions = {'C': [0.001, 0.01, 0.1, 1, 10]}
    grid = RandomizedSearchCV(LogisticRegression(max_iter=1000, solver='liblinear'), 
                               param_distributions=param_distributions, 
                               n_iter=3, cv=2, random_state=42, n_jobs=-1)
    grid.fit(X_train_tfidf, y_train)
    print("Best cross-validation score: {:.2f}".format(grid.best_score_))
    print("Best parameters: ", grid.best_params_)

    # Predict on multiple new reviews
    new_reviews = [
        "This product is fantastic! Highly recommend.",
        "Terrible experience. Not worth the money.",
        "Good quality, but could be cheaper.",
        "Absolutely loved it! Will buy again.",
        "Not bad, but shipping was slow."
    ]

    for review in new_reviews:
        new_review_cleaned = preprocess_text(review, stop_words)
        new_review_tfidf = tfidf.transform([new_review_cleaned])
        predicted_score = model.predict(new_review_tfidf)
        predicted_score = np.clip(np.round(predicted_score), y.min(), y.max())
        print(f"Review: {review}\nPredicted Score: {predicted_score[0]:.2f}\n")

if __name__ == "__main__":
    main()


[nltk_data] Downloading package stopwords to /Users/reza/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Data loaded successfully with shape: (568454, 10)
Mean Squared Error: 1.08
Accuracy: 0.73
Best cross-validation score: 0.73
Best parameters:  {'C': 10}
Review: This product is fantastic! Highly recommend.
Predicted Score: 5.00

Review: Terrible experience. Not worth the money.
Predicted Score: 1.00

Review: Good quality, but could be cheaper.
Predicted Score: 5.00

Review: Absolutely loved it! Will buy again.
Predicted Score: 5.00

Review: Not bad, but shipping was slow.
Predicted Score: 4.00



In [11]:
# Initialize stop_words globally
stop_words = set(stopwords.words('english'))

# New function for predicting new reviews
def predict_reviews(new_reviews, model, tfidf, stop_words):
    """Predict scores for a list of new reviews."""
    results = []
    for review in new_reviews:
        cleaned_review = preprocess_text(review, stop_words)
        review_tfidf = tfidf.transform([cleaned_review])
        predicted_score = model.predict(review_tfidf)
        predicted_score = np.clip(np.round(predicted_score), 1, 5)  # Adjust range if necessary
        results.append((review, predicted_score[0]))
    return results

# Main function to execute the pipeline
def main():
    # File path to dataset
    file_path = "/Users/reza/Desktop/ML/amazon/Reviews.csv"
    
    # Load the data
    df = load_data(file_path)
    if df is None:
        return

    # Check if required columns exist
    if 'Text' not in df.columns or 'Score' not in df.columns:
        print("Dataset does not contain the required columns: 'Text' and 'Score'")
        return

    # Preprocess text data
    df['cleaned_review'] = df['Text'].apply(lambda x: preprocess_text(x, stop_words))

    # Define features and target
    X = df['cleaned_review']
    y = df['Score']

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Vectorize the text data
    tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    # Train a logistic regression model
    model = LogisticRegression(max_iter=1000, solver='liblinear')
    model.fit(X_train_tfidf, y_train)

    # Predict and evaluate the model
    y_pred = model.predict(X_test_tfidf)

    # Round and clip predictions to match target labels
    y_pred_classes = np.round(y_pred).astype(int)
    y_pred_classes = np.clip(y_pred_classes, y.min(), y.max())

    # Compute metrics
    mse = mean_squared_error(y_test, y_pred_classes)
    accuracy = accuracy_score(y_test, y_pred_classes)
    print(f"Mean Squared Error: {mse:.2f}")
    print(f"Accuracy: {accuracy:.2f}")

    # Return the model, tfidf, and stop_words
    return model, tfidf, stop_words

# Run the main function and make components reusable
if __name__ == "__main__":
    model, tfidf, stop_words = main()

# Example: Run this in a separate cell to predict reviews
# """
# new_reviews = [
#     "This product is fantastic! Highly recommend.",
#     "Terrible experience. Not worth the money.",
#     "Good quality, but could be cheaper.",
#     "Absolutely loved it! Will buy again.",
#     "Not bad, but shipping was slow."
# ]

# results = predict_reviews(new_reviews, model, tfidf, stop_words)
# for review, score in results:
#     print(f"Review: {review}\nPredicted Score: {score:.2f}\n")
# """


Data loaded successfully with shape: (568454, 10)
Mean Squared Error: 1.08
Accuracy: 0.73


'\nnew_reviews = [\n    "This product is fantastic! Highly recommend.",\n    "Terrible experience. Not worth the money.",\n    "Good quality, but could be cheaper.",\n    "Absolutely loved it! Will buy again.",\n    "Not bad, but shipping was slow."\n]\n\nresults = predict_reviews(new_reviews, model, tfidf, stop_words)\nfor review, score in results:\n    print(f"Review: {review}\nPredicted Score: {score:.2f}\n")\n'

In [27]:
new_reviews = [
    "This product is bad! not recommend.",
    "Terrible experience. Not worth the money.",
    "Good quality, but could be cheaper.",
    "Absolutely loved it! Will buy again.",
    "as i expected"
]

results = predict_reviews(new_reviews, model, tfidf, stop_words)
for review, score in results:
    print(f"Review: {review}\nPredicted Score: {score:.2f}\n")


Review: This product is bad! not recommend.
Predicted Score: 1.00

Review: Terrible experience. Not worth the money.
Predicted Score: 1.00

Review: Good quality, but could be cheaper.
Predicted Score: 5.00

Review: Absolutely loved it! Will buy again.
Predicted Score: 5.00

Review: as i expected
Predicted Score: 3.00

